# HIERARCHY with AI agents – in Google Colab

Seven AI agents play the leadership exercise HIERARCHY. This notebook runs everything in your browser: no installation needed.

**Steps:** get the code → install → run the free answer key → add your API key → run a game → download the logs and open them in `replay.html` on your own computer.

Run the cells one at a time, from top to bottom (Shift + Enter). Colab forgets everything when the session ends, so download your logs before you close the tab.

## 1. Get the code

Run the cell. It downloads the kit from GitHub into this Colab session.

In [ ]:
import os, glob, subprocess
from google.colab import files

REPO = "https://github.com/YetiKetil/hierarchy_agents"
FOLDER = "/content/hierarchy_agents"
if not os.path.exists(FOLDER):
    subprocess.run(["git", "clone", "-q", REPO, FOLDER], check=True)
os.chdir(FOLDER)
print("Working in:", os.getcwd())

## 2. Install the libraries

In [ ]:
!pip install -q -r requirements.txt

## 3. Free: the answer key

An algorithm with full overview of all the cards solves the task under the same rules as the agents. No API key needed.

In [ ]:
!python run.py betingelser/en/alle_kjenner_maalet.yaml --answer-key

## 4. Add your API key

1. Click the **key icon** (Secrets) in the left sidebar.
2. Add a secret called `GEMINI_API_KEY` (or `ANTHROPIC_API_KEY` for Claude) and paste your key as the value.
3. Switch on *Notebook access* for the secret, and run the cell below.

Your key stays in your own Colab account and is not saved in the notebook.

In [ ]:
from google.colab import userdata
for name in ("GEMINI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        value = userdata.get(name) or ""
    except Exception:
        value = ""
    lines = [l.strip() for l in value.splitlines() if l.strip()]
    if not lines:
        print(name, "not set")
        continue
    os.environ[name] = lines[0]
    print(name, "found")
    if len(lines) > 1:
        print("  Warning: the secret has more than one line. Only the first line is used – check the value in Secrets.")

## 5. A short trial game

Ten rounds. **Choose your model in the first lines of the cell:** Gemini is selected; for Claude, put a `#` in front of the Gemini line and remove the `#` in front of the Claude line. The model you choose here is also used in step 6. On a free tier, keep `--concurrent 1`.

Each line is one round. *Shortfall* is the number of cards the group is missing before everyone has four of a kind (0 = solved). If something is wrong with the model name or the key, an **API ERROR** line tells you what the provider said.

In [ ]:
MODEL = "gemini-3.5-flash-lite"      # Gemini
# MODEL = "claude-haiku-5-5"         # Claude

!python run.py betingelser/en/prosedyre_lappet.yaml --model {MODEL} --max-rounds 10 --concurrent 1

## 6. A full game

Up to 60 rounds; this takes a while. Try another condition by changing `CONDITION`:
`grunnbetingelse` (only A knows the goal), `alle_kjenner_maalet`, `intensjon`, `prosedyre` or `prosedyre_lappet`.

In [ ]:
CONDITION = "prosedyre_lappet"
!python run.py betingelser/en/{CONDITION}.yaml --model {MODEL} --concurrent 1

## 7. Look at the results

In [ ]:
!python summarize.py --en

In [ ]:
# The latest game as text, including the agents' private thoughts (first 120 lines)
latest = max(glob.glob("logger/*/*.json"), key=os.path.getmtime)
print(latest)
!python show_game.py "{latest}" --thoughts 2>/dev/null | head -120

## 8. Download your logs

Download the zip file, unpack it on your own computer, and open the `.json` files in `replay.html`.

In [ ]:
import shutil
shutil.make_archive("my_hierarchy_logs", "zip", "logger")
files.download("my_hierarchy_logs.zip")

## Going further

`ADVANCED.md` explains how the program works, how to write your own conditions in `betingelser/en/`, and how to add other model providers. To edit a condition here in Colab, open the file browser (folder icon on the left) and double-click the file.